# Train Alpaca-Cleaned in Colab (instruction dataset)

Ce notebook prepare Alpaca-Cleaned au format instruction, tokenize le corpus, puis lance l'entrainement.

Note: Alpaca-Cleaned est un dataset d'instruction tuning.

In [ ]:
!nvidia-smi || true
!pip -q install datasets

In [ ]:
!git clone https://github.com/your-username/transformer-lab.git 2>/dev/null || true
%cd /content/transformer-lab

In [ ]:
from pathlib import Path
from datasets import load_dataset

DATA_DIR = Path('/content/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_JSONL = DATA_DIR / 'alpaca_cleaned_train.jsonl'
VAL_JSONL = DATA_DIR / 'alpaca_cleaned_val.jsonl'

if not TRAIN_JSONL.exists() or not VAL_JSONL.exists():
    ds = load_dataset('yahma/alpaca-cleaned', split='train')
    splits = ds.train_test_split(test_size=0.02, seed=42)
    splits['train'].to_json(str(TRAIN_JSONL))
    splits['test'].to_json(str(VAL_JSONL))

print(f'train_instruction_path={TRAIN_JSONL}')
print(f'val_instruction_path={VAL_JSONL}')

In [ ]:
TOKEN_DIR = '/content/data/alpaca_cleaned_tokens'
!python -m src.data.loader \
  --dataset_type instruction \
  --train_instruction_path /content/data/alpaca_cleaned_train.jsonl \
  --val_instruction_path /content/data/alpaca_cleaned_val.jsonl \
  --instruction_field instruction \
  --input_field input \
  --output_field output \
  --output_dir /content/data/alpaca_cleaned_tokens

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/transformer_checkpoints_alpaca_cleaned'

In [ ]:
!python -m src.training.trainer \
  --dataset_type instruction \
  --token_metadata_path /content/data/alpaca_cleaned_tokens/meta.json \
  --seq_len 256 \
  --num_epochs 3 \
  --batch_size 32 \
  --d_model 384 \
  --n_heads 6 \
  --n_layers 8 \
  --learning_rate 2e-4 \
  --checkpoint_dir /content/drive/MyDrive/transformer_checkpoints_alpaca_cleaned